# 🏥 Prescription NER — BioBERT Training on Google Colab

**Steps:**
1. Upload your CoNLL files
2. Install packages
3. Train BioBERT
4. Download trained model
5. Test inference

> ⚡ Make sure GPU is enabled: **Runtime → Change runtime type → T4 GPU**

## ✅ Step 0 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
# Should print: GPU name: Tesla T4

## 📦 Step 1 — Install Packages

In [ ]:
!pip install transformers torch tqdm seqeval -q
print('✅ All packages installed!')

## 📁 Step 2 — Upload Your CoNLL Files

Upload these 3 files from your laptop:
- `train.conll` (from `data/custom_indian/train.conll`)
- `dev.conll`
- `test.conll`

In [ ]:
from google.colab import files
import os

os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/model_output', exist_ok=True)

print('📂 Please upload your 3 CoNLL files: train.conll, dev.conll, test.conll')
uploaded = files.upload()

# Move files to /content/data/
for fname in uploaded:
    os.rename(fname, f'/content/data/{fname}')
    print(f'✅ Moved {fname} → /content/data/{fname}')

print('\nFiles in /content/data/:')
print(os.listdir('/content/data'))

## 🏷️ Step 3 — Define Labels

In [ ]:
# Your 9 NER labels (from models/label_map.json)
LABEL2ID = {
    'O': 0,
    'B-FORM': 1,
    'B-DRUG_BRAND': 2,
    'B-DRUG_GENERIC': 3,
    'B-DOSAGE': 4,
    'I-DOSAGE': 5,
    'B-FREQUENCY': 6,
    'B-DURATION': 7,
    'I-DURATION': 8
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

print(f'✅ {NUM_LABELS} labels defined')
print(ID2LABEL)

## 📖 Step 4 — Load CoNLL Data

In [ ]:
def parse_conll_file(file_path):
    """Reads CoNLL file and returns list of (tokens, labels) tuples."""
    examples = []
    tokens, labels = [], []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:  # blank line = sentence boundary
                if tokens:
                    examples.append((tokens, labels))
                    tokens, labels = [], []
            elif not line.startswith('#'):  # skip comments
                parts = line.split()
                if len(parts) >= 2:
                    tokens.append(parts[0])
                    labels.append(parts[1])

    if tokens:  # last sentence
        examples.append((tokens, labels))

    return examples

train_data = parse_conll_file('/content/data/train.conll')
dev_data   = parse_conll_file('/content/data/dev.conll')
test_data  = parse_conll_file('/content/data/test.conll')

print(f'✅ Train sentences: {len(train_data)}')
print(f'✅ Dev sentences:   {len(dev_data)}')
print(f'✅ Test sentences:  {len(test_data)}')

# Show first example
print('\n📋 First training example:')
tokens, labels = train_data[0]
for t, l in zip(tokens, labels):
    print(f'  {t:<20} {l}')

## 🔤 Step 5 — Tokenize with BioBERT

In [ ]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import torch

MODEL_NAME = 'dmis-lab/biobert-base-cased-v1.2'
MAX_LEN = 128

print(f'⬇️  Downloading BioBERT tokenizer from HuggingFace...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('✅ Tokenizer loaded!')


class PrescriptionDataset(Dataset):
    def __init__(self, data, tokenizer, label2id, max_len=128):
        self.data = data
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, labels = self.data[idx]

        # Tokenize (handles subword splitting)
        encoding = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        # Align labels with subword tokens
        word_ids = encoding.word_ids()
        aligned_labels = []
        prev_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                aligned_labels.append(-100)  # special tokens ignored
            elif word_idx != prev_word_idx:
                aligned_labels.append(self.label2id.get(labels[word_idx], 0))
            else:
                aligned_labels.append(-100)  # subword tokens ignored
            prev_word_idx = word_idx

        # Pad to max_len
        aligned_labels = aligned_labels[:self.max_len]
        aligned_labels += [-100] * (self.max_len - len(aligned_labels))

        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels':         torch.tensor(aligned_labels, dtype=torch.long)
        }


train_dataset = PrescriptionDataset(train_data, tokenizer, LABEL2ID, MAX_LEN)
dev_dataset   = PrescriptionDataset(dev_data,   tokenizer, LABEL2ID, MAX_LEN)
test_dataset  = PrescriptionDataset(test_data,  tokenizer, LABEL2ID, MAX_LEN)

print(f'✅ Train dataset: {len(train_dataset)} examples')
print(f'✅ Dev dataset:   {len(dev_dataset)} examples')

# Show one tokenized example
sample = train_dataset[0]
print('\n🔍 Sample input_ids shape:', sample['input_ids'].shape)
print('🔍 Sample labels shape:   ', sample['labels'].shape)

## 🧠 Step 6 — Load BioBERT Model & Fine-tune

In [ ]:
from transformers import AutoModelForTokenClassification, get_linear_schedule_with_warmup
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm import tqdm

# --- Hyperparameters ---
BATCH_SIZE    = 16      # GPU can handle 16 easily
NUM_EPOCHS    = 5
LEARNING_RATE = 2e-5
PATIENCE      = 3       # Early stopping patience

# --- Device ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  Using device: {device}')

# --- Load model ---
print(f'⬇️  Loading BioBERT model...')
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
model.to(device)
print('✅ Model loaded!')

# --- DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# --- Optimizer & Scheduler ---
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=100, num_training_steps=total_steps
)

print(f'\n📊 Training config:')
print(f'  Batch size:    {BATCH_SIZE}')
print(f'  Epochs:        {NUM_EPOCHS}')
print(f'  Learning rate: {LEARNING_RATE}')
print(f'  Total steps:   {total_steps}')

In [ ]:
import os

best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'val_loss': []}

for epoch in range(NUM_EPOCHS):
    print(f'\n{'='*50}')
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}')
    print(f'{'='*50}')

    # ---- TRAINING ----
    model.train()
    total_train_loss = 0

    for batch in tqdm(train_loader, desc='Training'):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    history['train_loss'].append(avg_train_loss)
    print(f'  Train Loss: {avg_train_loss:.4f}')

    # ---- VALIDATION ----
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in tqdm(dev_loader, desc='Validation'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_val_loss += outputs.loss.item()

    avg_val_loss = total_val_loss / len(dev_loader)
    history['val_loss'].append(avg_val_loss)
    print(f'  Val Loss:   {avg_val_loss:.4f}')

    # ---- SAVE BEST MODEL ----
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        model.save_pretrained('/content/model_output')
        tokenizer.save_pretrained('/content/model_output')
        print(f'  ✅ Best model saved! (val_loss={best_val_loss:.4f})')
    else:
        patience_counter += 1
        print(f'  ⚠️  No improvement ({patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print('  🛑 Early stopping triggered!')
            break

print('\n🎉 Training complete!')
print(f'Best validation loss: {best_val_loss:.4f}')

## 📊 Step 7 — Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train Loss', marker='o')
plt.plot(history['val_loss'],   label='Val Loss',   marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BioBERT Training Loss')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('/content/training_curve.png', dpi=150)
plt.show()
print('✅ Plot saved!')

## 🧪 Step 8 — Test Inference on Sample Text

In [ ]:
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch, json

# Load best saved model
inf_tokenizer = AutoTokenizer.from_pretrained('/content/model_output')
inf_model     = AutoModelForTokenClassification.from_pretrained('/content/model_output')
inf_model.eval()

def predict(text):
    inputs = inf_tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        outputs = inf_model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)[0].tolist()
    tokens = inf_tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Group entities
    entities = {}
    current_label, current_tokens = None, []

    for token, pred_id in zip(tokens, predictions):
        if token in ['[CLS]', '[SEP]', '[PAD]']:
            continue
        label = ID2LABEL.get(pred_id, 'O')
        if label.startswith('B-'):
            if current_label:
                word = ''.join(current_tokens).replace('##', '')
                entities.setdefault(current_label, []).append(word)
            current_label = label[2:]
            current_tokens = [token]
        elif label.startswith('I-') and current_label:
            current_tokens.append(token)
        else:
            if current_label:
                word = ''.join(current_tokens).replace('##', '')
                entities.setdefault(current_label, []).append(word)
            current_label, current_tokens = None, []

    if current_label:
        word = ''.join(current_tokens).replace('##', '')
        entities.setdefault(current_label, []).append(word)

    return entities

# Test it!
test_text = 'Tab Crocin 500mg BD x 3 days Cap Dolo 650 TDS'
result = predict(test_text)
print(f'Input: "{test_text}"')
print('\n📋 Extracted Entities:')
print(json.dumps(result, indent=2))

## 💾 Step 9 — Download Trained Model to Your Laptop

In [ ]:
import shutil
from google.colab import files

# Zip the model folder
shutil.make_archive('/content/biobert_prescription', 'zip', '/content/model_output')
print('✅ Model zipped!')

# Download to your laptop
files.download('/content/biobert_prescription.zip')
print('⬇️  Download started!')
print('\nAfter downloading, unzip and place the folder at:')
print('C:/Users/HP/Desktop/Prescription_Reader/models/biobert_prescription/')